In [ ]:
# download the weaviate client
%pip install -U weaviate-client

In [ ]:
import weaviate, os
from weaviate.config import AdditionalConfig, Timeout
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve environment variables
CLUSTER_URL = os.getenv("CLUSTER_URL")
API_KEY = os.getenv("API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
	cluster_url=CLUSTER_URL,
	auth_credentials=weaviate.auth.AuthApiKey(API_KEY),
	headers={
		"X-OpenAI-Api-Key": OPENAI_API_KEY,
		"X-Cohere-Api-Key": COHERE_API_KEY,
        "X-Goog-Api-Key": GOOGLE_API_KEY
	},
	additional_config=AdditionalConfig(
		timeout=Timeout(init=30, query=60, insert=120)
	)
)

ready = client.is_ready()
server_version = client.get_meta()["version"]
client_version = weaviate.__version__
live = client.is_live()
connected = client.is_connected()

print(f"Weaviate Ready: {ready}")
print(f"Weaviate Client Version: {client_version}")
print(f"Weaviate Server Version: {server_version}")
print(f"Weaviate Live: {client.is_live()}")
print(f"Client Connected: {connected}")

In [ ]:
collection = client.collections.use("<COLLECTION_NAME>")

for item in collection.iterator(
    include_vector=True
):
    print(item.uuid)
    print(item.properties)
    print(item.vector) 

In [ ]:
# List the collection names and their dimensions in Weaviate cluster
try:
    collections = client.collections.list_all()
    if collections:
        print("Collections and their dimensionality in Weaviate:")
        
        # Loop through each collection in the instance
        for collection_name in collections.keys():         
            coll = client.collections.use(collection_name)           
            try:
                # Try checking if the collection has tenants (Multi-Tenancy enabled)
                try:
                    tenants = coll.tenants.get()
                    is_multi_tenant = True
                except Exception:
                    is_multi_tenant = False 
                if is_multi_tenant:
                    # Multi-Tenant Collection: Get first available tenant
                    if tenants:
                        first_tenant = list(tenants.values())[0]  # Get first tenant object
                        tenant_name = first_tenant.name  # Extract tenant name
                        # Get the collection specific to this tenant
                        tenant_coll = coll.with_tenant(tenant_name)
                        # Fetch an object for the tenant
                        some_data = tenant_coll.query.fetch_objects(include_vector=True, limit=1)
                    else:
                        print(f"Collection: {collection_name} - No tenants found.")
                        continue
                else:
                    # Standard Collection (Non-MT)
                    some_data = coll.query.fetch_objects(include_vector=True, limit=1) 
                # Process fetched object
                if some_data.objects:
                    default_vector = some_data.objects[0].vector["default"]
                    dimensionality = len(default_vector)
                    print(f"Collection: {collection_name} - Dimensionality: {dimensionality}")
                else:
                    print(f"Collection: {collection_name} - No objects found.")
            except Exception as inner_e:
                print(f"Collection: {collection_name} - Error fetching objects: {inner_e}")
    else:
        print("No collections found.")
except Exception as e:
    print(f"Error retrieving collections or dimensions: {e}")

In [ ]:
# List the collection and its count
try:
    collections = client.collections.list_all()
    if collections:
        print("Collections in Weaviate:")
        print(f"Total collections: {len(collections)}")
    else:
        print("No collections found.")
except Exception as e:
    print(f"Error retrieving collections: {e}")

In [ ]:
# Retrieve the collection objects including the vectors
collection = client.collections.use("<COLLECTION_NAME>")
for item in collection.iterator(
    include_vector=True
):
    print(item.properties)
    print(item.vector)

In [ ]:
# Get the configuration of a specific collection
collection = client.collections.use("<COLLECTION_NAME>")
config = collection.config.get()
print(config.to_dict())

In [ ]:
# Get the configuration of a specific collection
schema_config = client.collections.list_all()
print(schema_config)